In [1]:
from scipy.io import loadmat
import numpy as np
import torch

device = torch.device('cuda') if torch.cuda.is_available() else torch.device("cpu")

In [2]:
image_display_time_s = 4
blank_display_time_s = 2
fixation_only_display_time_s = 2

n_classes = 3
batch_size = 16

## Merging datasets for all subjects

Dataset 1 from BCI IV (BBCI) contains data for 7 subjects. For each subject two classes of motor imagery were selected from the three classes left hand, right hand, and foot.

In [3]:
def get_cls_mapping_from_names(cls_names):
    # left: 0, right: 1, foot: 2
    name_to_num = {
        "left": 0,
        "right": 1,
        "foot": 2
    }
    names = [str(name[0]) for name in cls_names]
    return {
        -1: name_to_num.get(names[0]),
        1: name_to_num.get(names[1])
    }

subjects = ["a", "b", "c", "d", "e", "f", "g"]
file_paths = [f"data/BCICIV_calib_ds1{subject}_1000Hz.mat" for subject in subjects]
mat_a = loadmat(file_paths[0])
freq = mat_a["nfo"]["fs"][0][0][0][0]
trial_len = int(freq * image_display_time_s)  # 4 seconds

X_trials = []
y_trials = []
for file in file_paths:
    mat = loadmat(file)
    cls_names = mat["nfo"]["classes"][0][0][0]
    eeg_signal = mat["cnt"]
    target_classes = mat["mrk"]["y"][0][0][0]
    image_show_index = mat["mrk"]["pos"][0][0][0]
    for pos, cls in zip(image_show_index, target_classes):
        start = pos
        end = pos + trial_len
        if end <= eeg_signal.shape[0]:
            X_trials.append(eeg_signal[start:end])  # (T, C)
            cls_map = get_cls_mapping_from_names(cls_names)
            y_trials.append(cls_map.get(cls))


X_trials = torch.tensor(np.stack(X_trials)).float()  # (N_trials, T, C)
y_trials = torch.tensor(y_trials).long()

mean = X_trials.mean(dim=(0, 1), keepdim=True)
std = X_trials.std(dim=(0, 1), keepdim=True)
X_trials = (X_trials - mean) / (std + 1e-6)

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_trials, y_trials, test_size=0.2)

In [5]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [6]:
print("EEG signal:")
print(X_trials.shape)
print()
print("Classes corresponding to signal")
print(set(y_trials.numpy()))

EEG signal:
torch.Size([1400, 4000, 59])

Classes corresponding to signal
{np.int64(0), np.int64(1), np.int64(2)}


## Training

In [10]:
from torchesn.nn import ESN
import torch.nn.functional as F
from sklearn.metrics import confusion_matrix

n_channels = X_trials.shape[-1]

model = ESN(
    input_size=n_channels,
    hidden_size=512,
    num_layers=3,
    output_size=n_classes,
    # output_steps="mean",
    output_steps="last",
    readout_training='cholesky',
    batch_first=True
).to(device)

transient_period = int(freq * 1) # 1 second
for x, y in train_dataloader:
    x = x.to(device)
    y = y.to(device)
    washout = torch.full((x.size(0),), transient_period, device=device) # ignore first 1s
    model(x, washout, None, F.one_hot(y, n_classes).float())

model.fit()

### Train accuracy

In [11]:
train_preds = []
train_labels = []
for x, y in train_dataloader:
    with torch.no_grad():
        y_pred, _ = model(x, washout)
        pred_classes = y_pred[:, -1].argmax(dim=-1)
        train_labels.extend(y)
        train_preds.extend(pred_classes)

accuracy = (np.array(train_preds) == np.array(train_labels)).mean()
cm = confusion_matrix(train_labels, train_preds)
print(f"Accuracy: {accuracy.item():.4f}")
print(cm)

Accuracy: 0.8911
[[490  51  16]
 [ 51 347   0]
 [  4   0 161]]


### Test accuracy

In [12]:
test_dataset = TensorDataset(X_test, y_test)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size)

test_preds = []
test_labels = []
for x, y in test_dataloader:
    with torch.no_grad():
        y_pred, _ = model(x, washout)
        pred_classes = y_pred[:, -1].argmax(dim=-1)
        test_labels.extend(y)
        test_preds.extend(pred_classes)

accuracy = (np.array(test_preds) == np.array(test_labels)).mean()
cm = confusion_matrix(test_labels, test_preds)
print(f"Accuracy: {accuracy.item():.4f}")
print(cm)

Accuracy: 0.4786
[[63 55 25]
 [50 52  0]
 [15  1 19]]
